# Import de PySpark et des clés

In [3]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import numpy as np
from datetime import datetime, date

import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("BGES") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

# Tables
- FAITS MISSION
    - ID_PERSONNEL
    - ID_MISSION
- FAITS MATERIEL INFORMATIQUE
    - ID_PERSONNEL
    - ID_MATERIELINFO

- DATE : ANNEE, MOIS, JOUR
- SITE : SITE
- MISSION : ID_MISSION
- LOCALISATION : Ville
- Personnel - ID_PERSONNEL
- MATERIEL INFORMATIQUE - ID_MATERIELINFO

In [4]:
FAITS_MISSION = ps.read_csv('TABLES/FAITS_MISSION.csv',sep=',').to_spark()
FAITS_MATERIEL_INFORMATIQUE = ps.read_csv('TABLES/FAITS_MATERIEL_INFORMATIQUE.csv',sep=',').to_spark()
DATE = ps.read_csv('TABLES/DATE.csv',sep=',').to_spark()
SITE = ps.read_csv('TABLES/SITE.csv',sep=',').to_spark()
MISSION = ps.read_csv('TABLES/MISSION.csv',sep=',').to_spark()
PERSONNEL = ps.read_csv('TABLES/PERSONNEL.csv',sep=',').to_spark()
MATERIEL_INFORMATIQUE = ps.read_csv('TABLES/MATERIEL_INFORMATIQUE.csv',sep=',').to_spark()
LOCALISATION = ps.read_csv('TABLES/LOCALISATION.csv',sep=',').to_spark()

C:\Users\vdugr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
C:\Users\vdugr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [5]:
# Choix de traduction : 
#Mapping des rôles et des types de mission pour les uniformiser dans la base de données
role_mapping = {
    #Economy
    'Ökonom': 'Economist',
    'Economiste': 'Economist',
    'Economist': 'Economist',

    #Engineer
    'Dateningenieur': 'Data Engineer',
    'Ingénieur Data': 'Data Engineer',
    'Data Engineer': 'Data Engineer',

    'Computeringenieur': 'Computer Engineer',
    'Ingénieur Informaticien': 'Computer Engineer',
    'Computer Engineer': 'Computer Engineer',

    #RH
    'Personalleiter': 'HR Manager',
    'DRH': 'HR Manager',
    'HRD': 'HR Manager',

    #Management
    'Führungskraft': 'Manager',
    'Leadership': 'Manager',
    'Cadre': 'Manager',  # choix le plus cohérent globalement

    #Executive
    'Business Executive': 'Business Executive'
}

## Requête sur le DataWarehouse

#### 1. Combien de cadres travaillent sur le site de Paris?

In [ ]:
#Récupérer toutes les associations sites/personnel
sites = FAITS_MISSION.select("ID_PERSONNEL", "SITE") \
    .unionByName(FAITS_MATERIEL_INFORMATIQUE.select("ID_PERSONNEL", "SITE"))

PERSONNEL \
    .join(sites,"ID_PERSONNEL","left") \
    .where(
        (col("FONCTION_PERSONNEL"=="Manager")) &
        (col("SITE")=="Paris")
    )\
    .select("ID_PERSONNEL") \
    .distinct() \
    .count()

Py4JError: An error occurred while calling z:org.apache.spark.sql.functions.col. Trace:
py4j.Py4JException: Method col([class java.lang.Boolean]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:342)
	at py4j.Gateway.invoke(Gateway.java:276)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:108)
	at java.base/java.lang.Thread.run(Thread.java:1583)



#### 2. Combien d’ingénieurs Data travaillent sur les sites aux États-Unis ?

In [9]:
PERSONNEL.where(
    (lower(col('PAYS'))=='usa') &
    (col('FONCTION_PERSONNEL')== 'Data Engineer')
)\
.select('ID_PERSONNEL')\
.distinct()\
.count()

2197

#### 3. Combien d’ingénieurs informaticiens travaillent dans l’organisation (tous sites compris) ?

In [11]:
PERSONNEL\
    .filter(col('FONCTION_PERSONNEL')=='Computer Engineer')\
    .count()

7696

#### 4. Combien de PC fixes ont été achetés par l’organisation entre juin et septembre 2026 ?

In [12]:
FAITS_MATERIEL_INFORMATIQUE \
    .join(MATERIEL_INFORMATIQUE, on="ID_MATERIELINFO") \
    .where((col("ANNEE") == 2026) & (col("MOIS").between(6, 9))) \
    .where(lower(trim(col("TYPE"))) == "pc fixe") \
    .count()

1423

#### 5. Quelle a été l’impact carbone des PC fixes sans ecran entre mai et octobre 2026 ?

In [13]:
impact_c = FAITS_MATERIEL_INFORMATIQUE \
    .join(MATERIEL_INFORMATIQUE,on="ID_MATERIELINFO")\
    .where(
        (col("ANNEE")==2026) &
        (col('MOIS').between(5,10)) &
        (lower(trim(col('TYPE'))) == 'pc fixe') &
        (col('ECRAN')=='non')
    )\
    .agg(sum(col('IMPACT')))\
    .collect()[0][0]

print(
f"L'impact carbone des pc fixes sans ecran \nentre mai et octobre 2026 est {impact_c} t/CO²")

L'impact carbone des pc fixes sans ecran 
entre mai et octobre 2026 est 480.9210000000007 t/CO²


#### 6. Quel a été l’impact carbone des PC portables achetés par les ingénieurs Data entre mai et octobre 2026 sur les sites de Londres et New-York?

In [15]:
impact_c = FAITS_MATERIEL_INFORMATIQUE \
    .join(MATERIEL_INFORMATIQUE,on="ID_MATERIELINFO")\
    .join(PERSONNEL,on='ID_PERSONNEL')\
    .where(
        (col("ANNEE")==2026) &
        (col('MOIS').between(5,10)) &
        (lower(trim(col('TYPE'))) == 'pc portable') &
        ((col('SITE')=='New-York') | (col('SITE')=='Londres')) &
        (col('FONCTION_PERSONNEL')=='Data Engineer') &
        (col('ECRAN')=='non')
    )\
    .agg(sum(col('IMPACT')))\
    .collect()[0][0]


print(f"L'impact carbone des pc fixes sans ecran "
      f"\nachetés entre mai et octobre 2026 par \n"
      f"des ingénieurs data sur les sites de Londres\n"
      f"et New-York est {impact_c} t/CO²")

L'impact carbone des pc fixes sans ecran 
achetés entre mai et octobre 2026 par 
des ingénieurs data sur les sites de Londres
et New-York est None t/CO²


#### 7. Quel a été l’impact carbone des écrans achetés par les cadres entre juillet et septembre 2026 sur tous les sites de l’organisation ?

In [16]:
impact_c = FAITS_MATERIEL_INFORMATIQUE \
    .join(MATERIEL_INFORMATIQUE, on="ID_MATERIELINFO") \
    .join(PERSONNEL, on="ID_PERSONNEL") \
    .where((col("ANNEE") == 2026) & (col("MOIS").between(7, 9))) \
    .where(col("ECRAN") == "oui") \
    .where(col("FONCTION_PERSONNEL") == "Manager") \
    .agg(sum(col("IMPACT"))) \
    .collect()[0][0]

print(f"L'impact carbone des écrans"
      f"\nachetés entre juillet et septembre 2026 par \n"
      f"des cadres est {impact_c} t/CO²")

L'impact carbone des écrans
achetés entre juillet et septembre 2026 par 
des cadres est 79.95499999999998 t/CO²


#### 8. Quel a été l’impact carbone des missions sur les sites Européens entre mai et octobre 2026 ?

In [38]:
impact_c = FAITS_MISSION\
    .join(MISSION, on='ID_MISSION')\
    .join(LOCALISATION.select(col('VILLE').alias('SITE'),col('CONTINENT')), on='SITE')\
    .where(
        (col('CONTINENT')=='Europe') &
        (col("ANNEE") == 2026) &
        (col("MOIS").between(5, 10))
    )\
    .agg(sum(col("IMPACT_CARBONE"))) \
    .collect()[0][0]

print(f"L'impact carbone des missions sur les sites européens"
      f"\nentre mai et octobre 2026 est {impact_c} t/CO²")

L'impact carbone des missions sur les sites européens
entre mai et octobre 2026 est 13377.037184085068 t/CO²


#### 9. Quels ont été les 5 jours les plus impactants concernant les missions en avion pour les sites Européens de l’organisation ?

In [30]:
FAITS_MISSION \
    .join(MISSION, on="ID_MISSION") \
    .where(col("SITE").isin("Paris", "London", "Berlin")) \
    .where(lower(trim(col("TRANSPORT"))) == "avion") \
    .groupBy("ANNEE", "MOIS", "JOUR") \
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_TOTAL")
    ) \
    .orderBy(desc("IMPACT_TOTAL")) \
    .show(5, truncate=False)

+-----+----+----+------------------+
|ANNEE|MOIS|JOUR|IMPACT_TOTAL      |
+-----+----+----+------------------+
|2026 |8   |5   |129.90543691485982|
|2026 |5   |30  |111.82489133502227|
|2026 |5   |15  |110.4721547557106 |
|2026 |11  |12  |109.83076201958202|
|2026 |5   |19  |109.79112542038887|
+-----+----+----+------------------+
only showing top 5 rows


#### 10. Quel a été le secteur d’activité qui a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

In [22]:
impact_missions = (
    FAITS_MISSION
    .join(MISSION, on="ID_MISSION")
    .join(PERSONNEL, on="ID_PERSONNEL")
    .groupBy("FONCTION_PERSONNEL")
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_MISSIONS")
    )
)

impact_materiel = (
    FAITS_MATERIEL_INFORMATIQUE
    .join(MATERIEL_INFORMATIQUE, on="ID_MATERIELINFO")
    .join(PERSONNEL, on="ID_PERSONNEL")
    .groupBy("FONCTION_PERSONNEL")
    .agg(
        sum("IMPACT").alias("IMPACT_MATERIEL")
    )
)

impact_total = (
    impact_missions
    .join(
        impact_materiel,
        on="FONCTION_PERSONNEL",
        how="outer"
    )
    .fillna(0)
    .withColumn(
        "IMPACT_TOTAL",
        col("IMPACT_MISSIONS") + col("IMPACT_MATERIEL")
    )
)

impact_total \
    .orderBy(desc("IMPACT_TOTAL")) \
    .show(1, truncate=False)

+------------------+------------------+-----------------+------------------+
|FONCTION_PERSONNEL|IMPACT_MISSIONS   |IMPACT_MATERIEL  |IMPACT_TOTAL      |
+------------------+------------------+-----------------+------------------+
|Data Engineer     |10792.673381757884|802.9609999999938|11595.634381757878|
+------------------+------------------+-----------------+------------------+
only showing top 1 row


#### 11.  Quel site a eu le plus d’impact concernant les missions et le matériel informatique sur l’ensemble des sites de l’organisation ?

In [23]:
impact_missions = (
    FAITS_MISSION
    .join(MISSION, on="ID_MISSION")
    .groupBy("SITE")
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_MISSIONS")
    )
)

impact_materiel = (
    FAITS_MATERIEL_INFORMATIQUE
    .join(MATERIEL_INFORMATIQUE, on="ID_MATERIELINFO")
    .groupBy("SITE")
    .agg(
        sum("IMPACT").alias("IMPACT_MATERIEL")
    )
)

impact_total = (
    impact_missions
    .join(
        impact_materiel,
        on="SITE",
        how="outer"
    )
    .fillna(0)
    .withColumn(
        "IMPACT_TOTAL",
        col("IMPACT_MISSIONS") + col("IMPACT_MATERIEL")
    )
)

impact_total \
    .orderBy(desc("IMPACT_TOTAL")) \
    .show(1, truncate=False)

+-----------+-----------------+-----------------+-----------------+
|SITE       |IMPACT_MISSIONS  |IMPACT_MATERIEL  |IMPACT_TOTAL     |
+-----------+-----------------+-----------------+-----------------+
|Los Angeles|5969.213730421668|424.5729999999993|6393.786730421667|
+-----------+-----------------+-----------------+-----------------+
only showing top 1 row


#### 12. Quel a été l’impact carbone des missions reliant chaque site (la ville de départ est un site de l’organisation et la ville d’arrivée également) durant le mois de septembre 2026 ?

In [34]:
sites = SITE.select(col("SITE").alias("VILLE_SITE"))

missions_sites = (
    FAITS_MISSION
    .join(MISSION, on="ID_MISSION")
    .where((col("ANNEE") == 2026) & (col("MOIS") == 9))
    .join(
        sites.withColumnRenamed("VILLE_SITE", "VILLE_DEPART"),
        on="VILLE_DEPART",
        how="inner"
    )
    .join(
        sites.withColumnRenamed("VILLE_SITE", "VILLE_DESTINATION"),
        on="VILLE_DESTINATION",
        how="inner"
    )
    .groupBy("VILLE_DEPART", "VILLE_DESTINATION")
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_TOTAL")
    )
)

missions_sites.show(truncate=False)

+------------+-----------------+--------------------+
|VILLE_DEPART|VILLE_DESTINATION|IMPACT_TOTAL        |
+------------+-----------------+--------------------+
|Paris       |London           |2.483027274886013   |
|Berlin      |New-York         |20.379443302123164  |
|Shanghai    |Berlin           |17.870263895012513  |
|Los Angeles |London           |19.962346673973975  |
|London      |Los Angeles      |19.962346673973975  |
|New-York    |Shanghai         |43.259507667783744  |
|Berlin      |Shanghai         |16.59381647394019   |
|Paris       |Berlin           |6.789171793068204   |
|Paris       |Shanghai         |25.3430576668447    |
|New-York    |London           |9.313422673557861   |
|Los Angeles |Berlin           |32.54531197221438   |
|Paris       |Los Angeles      |16.571700537476463  |
|Shanghai    |London           |23.763662110965313  |
|Los Angeles |New-York         |9.744543844542626   |
|New-York    |New-York         |0.004967999999999999|
|London      |Shanghai      

#### 13. Quel a été l’impact carbone des conférences en juillet 2026 pour les employés de Los Angeles ?

In [ ]:
FAITS_MISSION \
    .join(MISSION, on="ID_MISSION") \
    .join(PERSONNEL, on="ID_PERSONNEL") \
    .where((col("ANNEE") == 2026) & (col("MOIS") == 7)) \
    .where(col("TYPE_MISSION") == "Conference") \
    .where(col("SITE") == "Los Angeles") \
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_TOTAL")
    ) \
    .show()

+------------------+
|      IMPACT_TOTAL|
+------------------+
|172.86839876340326|
+------------------+



#### 14. Quel secteur d’activité a été le plus impactant pour les missions “conférences” entre mai et septembre 2026 ?

In [26]:
FAITS_MISSION \
    .join(MISSION, on="ID_MISSION") \
    .join(PERSONNEL, on="ID_PERSONNEL") \
    .where((col("ANNEE") == 2026) & (col("MOIS").between(5, 9))) \
    .where(col("TYPE_MISSION") == "Conference") \
    .groupBy("FONCTION_PERSONNEL") \
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_TOTAL")
    ) \
    .orderBy(desc("IMPACT_TOTAL")) \
    .show(1, truncate=False)

+------------------+------------------+
|FONCTION_PERSONNEL|IMPACT_TOTAL      |
+------------------+------------------+
|Data Engineer     |1646.0690970171183|
+------------------+------------------+
only showing top 1 row


#### 15. Quel a été l’âge moyen des employés Ingénieurs Data qui sont partis en formations entre juillet et septembre 2026 ?

In [27]:
FAITS_MISSION \
    .join(MISSION, on="ID_MISSION") \
    .join(PERSONNEL, on="ID_PERSONNEL") \
    .where((col("ANNEE") == 2026) & (col("MOIS").between(7, 9))) \
    .where(col("TYPE_MISSION") == "Training") \
    .where(col("FONCTION_PERSONNEL") == "Data Engineer") \
    .withColumn(
        "AGE",
        floor(datediff(current_date(), col("DATE_NAISSANCE")) / 365.25)
    ) \
    .agg(
        avg("AGE").alias("AGE_MOYEN")
    ) \
    .show()

+-----------------+
|        AGE_MOYEN|
+-----------------+
|59.35414165666266|
+-----------------+



#### 16. Quelle destination a été la plus impactante (en cumul) entre mai et octobre 2026 ?

In [28]:
FAITS_MISSION \
    .join(MISSION, on="ID_MISSION") \
    .where((col("ANNEE") == 2026) & (col("MOIS").between(5, 10))) \
    .groupBy("VILLE_DESTINATION") \
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_TOTAL")
    ) \
    .orderBy(desc("IMPACT_TOTAL")) \
    .show(1, truncate=False)

+-----------------+------------------+
|VILLE_DESTINATION|IMPACT_TOTAL      |
+-----------------+------------------+
|Wellington       |1712.4591561305324|
+-----------------+------------------+
only showing top 1 row


#### 17. Quelles ont été les trois catégories de missions les plus impactantes pour les cadres dans les sites Européens en mai 2026 ?

In [29]:
FAITS_MISSION \
    .join(MISSION, on="ID_MISSION") \
    .join(PERSONNEL, on="ID_PERSONNEL") \
    .join(LOCALISATION, MISSION["VILLE_DESTINATION"] == LOCALISATION["VILLE"]) \
    .where((col("ANNEE") == 2026) & (col("MOIS") == 5)) \
    .where(col("FONCTION_PERSONNEL") == "Manager") \
    .where(col("CONTINENT") == "Europe") \
    .groupBy("TYPE_MISSION") \
    .agg(
        sum("IMPACT_CARBONE").alias("IMPACT_TOTAL")
    ) \
    .orderBy(desc("IMPACT_TOTAL")) \
    .show(3, truncate=False)

+----------------+------------------+
|TYPE_MISSION    |IMPACT_TOTAL      |
+----------------+------------------+
|Internal Meeting|32.03954924850914 |
|Conference      |28.768488258442172|
|Business Meeting|28.12172912681226 |
+----------------+------------------+
only showing top 3 rows


*Les réponses des questions suivantes devront être sous forme de figures*
#### 18. Quelles ont été les 5 missions les plus impactantes sur le site de Paris ?

#### 19. Proposer une figure comparant l’impact carbone mensuel des missions en fonction du type de transport et sur chaque site.

#### 20. Proposer une figure illustrant l’impact carbone global mensuel de l’organisation.